## Research Notebook for PISA question and answers on different languages

## Libraries

In [1]:
import os, getpass

In [2]:
from dotenv import load_dotenv, find_dotenv

In [3]:
from openai import OpenAI

In [4]:
load_dotenv(find_dotenv(usecwd=True))  # finds .env from current working dir upward

True

In [5]:
# --- 0) Setup: imports & config
import os, re, time, json, math, random
from typing import Dict, List, Tuple, Any, Optional
import pandas as pd
import time

In [6]:
from tqdm import tqdm

## Configuration

In [7]:
client = OpenAI() 

In [8]:
MODEL_GPT = "gpt-5.1-2025-11-13"                    # <- replace with exact model id if different
# gpt-5-2025-08-07
# gpt-5.1-2025-11-13
# gpt-5.2-2025-12-11

In [9]:
SHEET_ID = "1QVPzB7uMwqJ6jCsHkwIILnXvDQIycpqkcV3bkiDpzyQ" 
WORKSHEET_NAME = "dataset"  # change if needed "dataset"
RESULTS_CSV = "llm_eval_results_GPT5.csv"
SAMPLE_PER_LANGUAGE = 1     # 5 per language
MAX_LANGUAGES = 43          # 43 languages total
SEED = 42

random.seed(SEED)

## Load data from Google Sheet

In [10]:
csv_url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/gviz/tq?tqx=out:csv&sheet={WORKSHEET_NAME}"
try:
    df = pd.read_csv(csv_url)
except Exception as e:
    raise RuntimeError(
        "Failed to read the Google Sheet via CSV export. "
        "Make sure the sheet is shared as 'Anyone with the link can view', "
        f"ID is correct, and tab name matches. Underlying error: {e}"
    )

expected_cols = {
    "qid","language","question","context","options","gold",
    "answer_type","category","difficulty","rationale","source"
}
missing = expected_cols - set(df.columns)
if missing:
    raise ValueError(f"Your sheet is missing columns: {sorted(missing)}")

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1075 entries, 0 to 1074
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   qid            1075 non-null   object
 1   language       1075 non-null   object
 2   language_code  1075 non-null   object
 3   question       1075 non-null   object
 4   context        1075 non-null   object
 5   options        1075 non-null   object
 6   gold           1075 non-null   object
 7   answer_type    1075 non-null   object
 8   category       1075 non-null   object
 9   difficulty     1075 non-null   object
 10  rationale      172 non-null    object
 11  source         1075 non-null   object
dtypes: object(12)
memory usage: 100.9+ KB


## Normalize & sample

In [12]:
df["language"] = df["language"].astype(str).str.strip()
# pick first MAX_LANGUAGES languages (sorted)
languages = sorted(df["language"].unique())[:MAX_LANGUAGES]

In [13]:
# sample up to SAMPLE_PER_LANGUAGE per language
sampled = (
    df[df["language"].isin(languages)]
    .groupby("language", sort=True, group_keys=False)
    .head(SAMPLE_PER_LANGUAGE)
    .reset_index(drop=True)
)
if sampled.empty:
    raise ValueError("No rows selected. Check your data.")

In [14]:
print(f"Selected {len(sampled)} rows across {sampled['language'].nunique()} languages.")
display(sampled[["qid","language","question","gold"]])

Selected 43 rows across 43 languages.


,qid,language,question,gold
0,q001,Albanian,Nëse Tania vendos të blejë makinën D dhe ta sh...,C
1,q001,Arabic,إذا قررت تهاني شراء السيارة D وإعادة بيعها بع...,C
2,q001,Azerbaijani / Azeri,Əgər Turan maşın D-ni almağa və üç il sonra əl...,C
3,q001,Basque,Taniak erabakitzen badu D autoa erostea eta ho...,C
4,q001,Bokmål,Omtrent hvor mye vil bruktprisen av bilen være...,C
5,q001,Bosnian,"Ako Tanja odluči da kupi automobil D, i prepro...",C
6,q001,Bulgarian,Ако Таня реши да купи кола Г и я препродаде сл...,C
7,q001,Catalan,Si la Tània decideix comprar el cotxe D i reve...,C
8,q001,Chinese,如果田妮决定购买汽车 D 并于三年后在良好车况下转售，那么这辆汽车的转售价格大约是多少 (z...,C
9,q001,Croatian,Ako Tanja odluči kupiti automobil D i preproda...,C


## Parse options

In [15]:
def parse_options(raw: str) -> Dict[str, Any]:
    """
    Parse multiple-choice options from a JSON-encoded string.

    Supported formats
    -----------------
    1) Simple labeled strings (original format):
        ["A) 1575", "B) 8925", "C) 9000", "D) 9975"]

        -> {"A": "1575", "B": "8925", "C": "9000", "D": "9975"}

    2) List of dicts with labels mapping to lists of tokens:
        [
          {"A": ["India", "Colombia"]},
          {"B": ["India", "Armenia"]},
          {"C": ["Panama", "Colombia"]},
          {"D": ["Kazakhstan", "Colombia"]}
        ]
    """
    if pd.isna(raw):
        raise ValueError("Options are empty")

    # Load JSON
    try:
        items = json.loads(raw)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON format for options: {e}")

    if not isinstance(items, list):
        raise ValueError("Expected a JSON list at top level")

    # Case 1: list of strings -> original behavior
    if all(isinstance(item, str) for item in items):
        options: Dict[str, Any] = {}
        for item in items:
            # Match patterns like "A) text", "B. text", or "C: text"
            if not isinstance(item, str):
                raise ValueError(f"Option is not a string: {item}")
            match = re.match(r"^\s*([A-Z])[\)\.\:]\s*(.+)$", item.strip())
            if match:
                label, text = match.groups()
                options[label.upper()] = text.strip()
            else:
                # Fallback: assign next available letter automatically
                next_label = chr(ord('A') + len(options))
                options[next_label] = item.strip()
        return options

    # Case 2: list of dicts like [{"A": [...]}, {"B": [...]}]
    if all(isinstance(item, dict) for item in items):
        options: Dict[str, Any] = {}
        for idx, d in enumerate(items):
            if len(d) != 1:
                raise ValueError(
                    f"Each dict must have exactly one key (label). Problem at index {idx}: {d}"
                )
            (label_raw, value) = next(iter(d.items()))
            if not isinstance(label_raw, str):
                raise ValueError(f"Label must be a string, got: {label_raw}")

            label = label_raw.strip().upper()
            if not re.fullmatch(r"[A-Z]", label):
                raise ValueError(f"Invalid option label '{label_raw}' at index {idx}")

            # Accept list or scalar; normalize scalars into single-element lists if needed
            if isinstance(value, list):
                options[label] = value
            else:
                options[label] = [value]

        return options

    # If we reach here, the list is mixed or has unsupported types
    raise ValueError(
        "Unsupported options format: expected list of strings or list of single-key dicts"
    )


In [16]:
test = parse_options('["A) 2018", "B) 2019", "C) 2020", "D) 2021"]')
print(test)

{'A': '2018', 'B': '2019', 'C': '2020', 'D': '2021'}


In [17]:
options_block = "\n".join([f"{k}. {v}" for k,v in test.items()])
print(options_block)

A. 2018
B. 2019
C. 2020
D. 2021


## Build prompt

In [18]:
def build_prompt(row: pd.Series, options: Dict[str,str]) -> str:
    """
    Builds prompt for MCQ.
    """
    options_block = "\n".join([f"{k}. {v}" for k,v in options.items()])

    return (
        f"{row['context']}\n"
        f"{row['question']}\n"
        f"{options_block}\n\n"
    )

In [19]:
answer_letter_regex = re.compile(r"<\s*([A-Z])\s*[.)]?\s*>")

def extract_letter(text: str, valid_letters: List[str]) -> str:
    """
    Extract the first single-letter A-Z token that is in valid_letters.
    """
    if not text:
        return ""
    # First line is the letter per our format; but still be defensive:
    first_line = text.splitlines()[0].strip().rstrip(".)").upper()
    # If first line is a single valid letter, use it
    if len(first_line) == 1 and first_line.upper() in valid_letters:
        return first_line.upper()
    # Else find any A-Z token
    m = answer_letter_regex.search(text.upper())
    if m and m.group(1) in valid_letters:
        return m.group(1)
    return ""

## LLM call wrapper

In [20]:
def llm_completion(
    prompt: str,
    model: Optional[str] = None,
    max_tokens: Optional[int] = None,
    system_prompt: str="Reply format: <LETTER>", #"", #
    retries: int = 2,
    backoff_seconds: float = 1.5,
    reasoning_effort: Optional[str] = None,     # e.g., "low"|"medium"|"high" (and some models support "none")
    reasoning_summary: Optional[str] = None,    # e.g., "auto" (or "concise"/"detailed" depending on model)
    **kwargs,
) -> Tuple[str, Dict[str, Any]]:
    """
    Call OpenAI Responses API and return (text, usage_dict).
    usage_dict contains: input_tokens, output_tokens, total_tokens (if present)
    """
    mdl = model or MODEL_GPT
    last_err = None

    for attempt in range(retries + 1):
        try:
            # Build arguments dynamically — include only if provided
            call_args = dict(
                model=mdl,
                instructions=system_prompt,
                input= prompt,
                stream=False
            )

            # ---- attach token limits if provided ----
            if max_tokens is not None:
                call_args["max_tokens"] = max_tokens

            # ---- NEW: reasoning controls ---- :contentReference[oaicite:4]{index=4}
            if reasoning_effort is not None or reasoning_summary is not None:
                r = {}
                if reasoning_effort is not None:
                    r["effort"] = reasoning_effort
                if reasoning_summary is not None:
                    r["summary"] = reasoning_summary
                call_args["reasoning"] = r

            # Merge other kwargs (e.g., stop, seed, etc.)
            for k, v in kwargs.items():
                if k not in call_args:
                    call_args[k] = v

            # Actual model call
            response = client.responses.create(**call_args)
            if response is None:
                return "No response from model.", {}
            
            text = (getattr(response, "output_text", "") or "").strip()

            # --- Extract usage safely across SDK versions ---
            usage_obj = getattr(response, "usage", None)

            def _get(obj, key, default=None):
                if obj is None:
                    return default
                if isinstance(obj, dict):
                    return obj.get(key, default)
                return getattr(obj, key, default)

            usage: Dict[str, Any] = {
                "input_tokens": None,
                "output_tokens": None,
                "reasoning_tokens": None,
                "effort": reasoning_effort,
                "summary": "",
            }

            if usage_obj is not None:
                # SDK objects often allow attribute access; dict-like in some contexts.
                
                usage["input_tokens"] = _get(usage_obj, "input_tokens")
                usage["output_tokens"] = _get(usage_obj, "output_tokens")

                out_details = _get(usage_obj, "output_tokens_details", {})
                usage["reasoning_tokens"] = _get(out_details, "reasoning_tokens")    

            if reasoning_summary is not None:
                try:
                    out_items = getattr(response, "output", None) or []
                    for item in out_items:
                        item_type = _get(item, "type")
                        if item_type == "reasoning":
                            summaries = _get(item, "summary", []) or []
                            parts = []
                            for s in summaries:
                                if _get(s, "type") == "summary_text":
                                    parts.append(_get(s, "text", "") or "")
                            usage["summary"] = "\n\n".join([p for p in parts if p]).strip()
                            break
                except Exception:
                    # keep meta["summary"] as empty if shape differs
                    pass            

            return text, usage

        except Exception as e:
            last_err = e
            if attempt < retries:
                time.sleep(backoff_seconds * (attempt + 1))
            else:
                raise

## Evaluation loop

In [51]:
def eval_rows(
        rows: pd.DataFrame, 
        cycle: int,
        model_tag: str,
        model_name: str, 
        results_path: str = RESULTS_CSV,
        sleep_s: float = 0.0, 
        retries: int = 2
    ) -> pd.DataFrame:
    results = []
    file_exists = os.path.exists(results_path)
    for i, row in tqdm(rows.iterrows(), total=len(rows), desc=f"Evaluating {model_tag} / {model_name}, cycle {cycle}"):
        qid = row["qid"]
        lang = row["language"]
        gold = str(row["gold"]).strip().upper()
        difficulty = row["difficulty"]

        # Parse options
        try:
            opts = parse_options(row["options"])
        except Exception as e:
            row_result = {
                "qid": qid,
                "language": lang,
                "pred": "",
                "gold": gold,
                "is_correct": False,
                "error": f"OptionsParseError: {e}",
                "raw": "",
                "difficulty": difficulty,
                "question": row["question"],
                "options_json": json.dumps(opts if 'opts' in locals() else {}, ensure_ascii=False),
                "model_tag": model_tag,
                "model_name": model_name,
                "cycle": cycle,
                "input_tokens": 0,
                "output_tokens": 0,
                "reasoning_tokens": 0,
                "effort": "Default",
                "summary": "",
            }
            results.append(row_result)

            # Save immediately
            pd.DataFrame([row_result]).to_csv(
                results_path,
                mode="a",
                header=not file_exists,
                index=False,
                encoding="utf-8"
            )
            file_exists = True
            continue

        valid_letters = sorted(list(opts.keys()))
        prompt = build_prompt(row, opts)

        # call model with simple retry
        raw = ""
        usage = {}
        err = ""
        for attempt in range(retries + 1):
            try:
                raw, usage = llm_completion(
                    prompt,
                    model=model_name,
                    reasoning_effort="none",     # <-- Optional. choose per run / per model
                    reasoning_summary="auto",   # <-- Optional. or None to disable summaries
                )
                break
            except Exception as e:
                err = f"{type(e).__name__}: {e}"
                if attempt < retries:
                    time.sleep(1.5 * (attempt + 1))
                else:
                    raw, usage = "", {}
        pred = extract_letter(raw, valid_letters)
        is_correct = (pred == gold)

        row_result = {
            "qid": qid,
            "language": lang,
            "pred": pred,
            "gold": gold,
            "is_correct": bool(is_correct),
            "error": err,
            "raw": raw,
            "difficulty": difficulty,
            "question": row["question"],
            "options_json": json.dumps(opts, ensure_ascii=False),
            "model_tag": model_tag,
            "model_name": model_name,
            "cycle": cycle,
            "input_tokens": usage.get("input_tokens"),
            "output_tokens": usage.get("output_tokens"),
            "reasoning_tokens": usage.get("reasoning_tokens"),
            "effort": usage.get("effort"),
            "summary": usage.get("summary", ""),
        }

        results.append(row_result)

        # *** Save this row immediately ***
        pd.DataFrame([row_result]).to_csv(
            results_path,
            mode="a",
            header=not file_exists,
            index=False,
            encoding="utf-8"
        )
        file_exists = True

        if sleep_s > 0:
            time.sleep(sleep_s)
    return pd.DataFrame(results)

## Execution

In [52]:
MODELS_TO_TEST = [
    ("GPT", MODEL_GPT)
]

In [53]:
# filtered_df = df[df["language"] == "English"]
sampled = df

In [65]:
N_CYCLES = 5  # repeat the same question to the same LLM

In [ ]:
if os.path.exists(RESULTS_CSV):
    existing_results = pd.read_csv(RESULTS_CSV)
    print(f"Loaded existing results from {RESULTS_CSV}: {len(existing_results)} rows")
else:
    existing_results = pd.DataFrame()
    print("No existing results file found. Starting fresh.")

for tag, model_name in MODELS_TO_TEST:
    print(f"\nEvaluating {tag} -> {model_name}")
    for cycle in range(2, N_CYCLES + 1):
        # Determine which qids are already done for this (tag, model_name, cycle)
        if existing_results.empty:
            # Nothing done yet at all
            rows_to_eval = sampled.copy()
        else:
            # Subset only rows already done for this (tag, model_name, cycle)
            subset = existing_results[
                (existing_results["model_tag"] == tag) &
                (existing_results["model_name"] == model_name) &
                (existing_results["cycle"] == cycle)
            ]

            if subset.empty:
                # No rows done yet for this model+cycle
                rows_to_eval = sampled.copy()
            else:
                # Use (qid, language) pairs as the key — more robust than qid alone
                done_pairs = set(zip(subset["qid"], subset["language"]))

                mask = ~sampled.apply(
                    lambda r: (r["qid"], r["language"]) in done_pairs,
                    axis=1
                )
                rows_to_eval = sampled[mask]

        if rows_to_eval.empty:
            print(f"  • cycle {cycle}/{N_CYCLES}: already complete, skipping")
            continue

        print(f"  • cycle {cycle}/{N_CYCLES}: evaluating {len(rows_to_eval)} questions")
        t0 = time.time()

        df_new = eval_rows(
            rows_to_eval,
            model_tag=tag,
            model_name=model_name,
            cycle=cycle,
            results_path=RESULTS_CSV
        )

        elapsed = time.time() - t0
        print(f"    Done in {elapsed:.1f}s, newly evaluated {len(df_new)} rows.")

        # Update in-memory copy so subsequent cycles/ models can see freshly written rows
        existing_results = pd.concat([existing_results, df_new], ignore_index=True)

Loaded existing results from llm_eval_results_GPT5.csv: 3225 rows

Evaluating GPT -> gpt-5.1-2025-11-13
  • cycle 5/5: evaluating 1075 questions


Evaluating GPT / gpt-5.1-2025-11-13, cycle 5: 100%|██████████| 1075/1075 [16:12<00:00,  1.11it/s]

    Done in 972.6s, newly evaluated 1075 rows.


In [67]:
res_df = pd.read_csv(RESULTS_CSV)
print(f"\nTotal results loaded: {len(res_df)}")


Total results loaded: 4300


## Results

In [68]:
overall_by_model = (
    res_df.groupby(["model_tag","model_name"])["is_correct"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct":"accuracy"})
    .sort_values("accuracy", ascending=False)
)
print("\nOverall accuracy by model:")
display(overall_by_model)


Overall accuracy by model:


,model_tag,model_name,accuracy
0,GPT,gpt-5.1-2025-11-13,0.786744


In [69]:
overall_by_question = (
    res_df.groupby(["qid"])["is_correct"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct":"accuracy"})
    .sort_values("accuracy", ascending=False)
)
print("\nOverall accuracy by question:")
display(overall_by_question)


Overall accuracy by question:


,qid,accuracy
4,q005,1.000000
8,q009,1.000000
5,q006,1.000000
24,q025,1.000000
21,q022,1.000000
20,q021,1.000000
19,q020,1.000000
16,q017,1.000000
15,q016,1.000000
23,q024,0.976744


In [70]:
overall_by_lang = (
    res_df.groupby(["language"])["is_correct"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct":"accuracy"})
    .sort_values("accuracy", ascending=False)
)
print("\nOverall accuracy by lang:")
display(overall_by_lang)


Overall accuracy by lang:


,language,accuracy
6,Bulgarian,0.85
41,Thai,0.85
22,Hungarian,0.85
35,Russian,0.85
12,Dutch,0.84
15,Finnish,0.83
34,Portuguese,0.82
11,Danish,0.82
19,German,0.82
3,Basque,0.81


In [71]:
by_question = (
    res_df.groupby(["model_tag","model_name","language","qid"])["is_correct"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct":"accuracy"})
    .sort_values(["accuracy"])
)

# print("\nAccuracy by question:")
# display(by_question)

accuracy_counts_total = (
    by_question["accuracy"]
    .value_counts()
    .rename_axis("accuracy")
    .reset_index(name="count")
    .sort_values("accuracy")
)

print("\nCount of total questions by accuracy:")
display(accuracy_counts_total)

accuracy_counts = (
    by_question
    .groupby(["model_tag", "model_name", "accuracy"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .sort_values(["model_tag", "model_name", "accuracy"])
)

print("\nCount of questions by accuracy:")
display(accuracy_counts)


Count of total questions by accuracy:


,accuracy,count
1,0.00,190
2,0.25,28
4,0.50,24
3,0.75,25
0,1.00,808



Count of questions by accuracy:


,model_tag,model_name,accuracy,count
0,GPT,gpt-5.1-2025-11-13,0.00,190
1,GPT,gpt-5.1-2025-11-13,0.25,28
2,GPT,gpt-5.1-2025-11-13,0.50,24
3,GPT,gpt-5.1-2025-11-13,0.75,25
4,GPT,gpt-5.1-2025-11-13,1.00,808


In [72]:
def safe_acc(s):
    return float('nan') if s.empty else s.mean()
overall_acc = safe_acc(res_df["is_correct"])
print(f"\nCombined overall accuracy on {len(res_df)} items: {overall_acc:.3f}")


Combined overall accuracy on 4300 items: 0.787


In [73]:
for tag, _ in MODELS_TO_TEST:
    out_path = f"llm_eval_results__{tag}.csv"
    res_df.query("model_tag == @tag").to_csv(out_path, index=False)
    print(f"Saved {tag} results to: {out_path}")

# (Optional) quick peek
display(res_df.head())

Saved GPT results to: llm_eval_results__GPT.csv


,qid,language,pred,gold,is_correct,error,raw,difficulty,question,options_json,model_tag,model_name,cycle,input_tokens,output_tokens,reasoning_tokens,effort,summary
0,q001,Albanian,B,C,False,NaN,B,level 6,Nëse Tania vendos të blejë makinën D dhe ta sh...,"{""A"": ""1575"", ""B"": ""8925"", ""C"": ""9000"", ""D"": ""...",GPT,gpt-5.1-2025-11-13,2,198,11,0,none,NaN
1,q002,Albanian,B,C,False,NaN,B,level 6,"Nëse trendi i shitjeve vazhdon, në cilin vit n...","{""A"": ""2015"", ""B"": ""2018"", ""C"": ""2020"", ""D"": ""...",GPT,gpt-5.1-2025-11-13,2,126,11,0,none,NaN
2,q003,Albanian,B,B,True,NaN,B,level 2,Cili është numri më i madh i kutive mesatare q...,"{""A"": ""320"", ""B"": ""128"", ""C"": ""26"", ""D"": ""16""}",GPT,gpt-5.1-2025-11-13,2,224,11,0,none,NaN
3,q004,Albanian,C,C,True,NaN,C,level 6,Kompania e cila jep kamionë me qera konfirmoi ...,"{""A"": ""Ajo ka të drejtë, sepse lartësia e një ...",GPT,gpt-5.1-2025-11-13,2,540,11,0,none,NaN
4,q005,Albanian,D,D,True,NaN,D,level 2,"Mesatarisht, afërisisht sa milionë kilometra g...","{""A"": ""5 milionë km"", ""B"": ""30 milionë km"", ""C...",GPT,gpt-5.1-2025-11-13,2,171,11,0,none,NaN


In [74]:
df_false = res_df[res_df['is_correct'] == False]
display(df_false.head(50))

,qid,language,pred,gold,is_correct,error,raw,difficulty,question,options_json,model_tag,model_name,cycle,input_tokens,output_tokens,reasoning_tokens,effort,summary
0,q001,Albanian,B,C,False,NaN,B,level 6,Nëse Tania vendos të blejë makinën D dhe ta sh...,"{""A"": ""1575"", ""B"": ""8925"", ""C"": ""9000"", ""D"": ""...",GPT,gpt-5.1-2025-11-13,2,198,11,0,none,NaN
1,q002,Albanian,B,C,False,NaN,B,level 6,"Nëse trendi i shitjeve vazhdon, në cilin vit n...","{""A"": ""2015"", ""B"": ""2018"", ""C"": ""2020"", ""D"": ""...",GPT,gpt-5.1-2025-11-13,2,126,11,0,none,NaN
9,q010,Albanian,C,A,False,NaN,C,level 6,Konsideroni dy periudhat kohore: 2005 deri në ...,"{""A"": [""India"", ""Kolumbia""], ""B"": [""India"", ""A...",GPT,gpt-5.1-2025-11-13,2,569,11,0,none,NaN
17,q018,Albanian,C,A,False,NaN,C,level 5,A është thënia fakt apo opinion?\nStudimet e f...,"{""A"": [""Opinion"", ""Fakt"", ""Fakt"", ""Opinion""], ...",GPT,gpt-5.1-2025-11-13,2,1630,11,0,none,NaN
18,q019,Albanian,C,A,False,NaN,C,level 3,A mund ta përfaqësojë kjo thënie qëllimin e ar...,"{""A"": [""Po"", ""Po"", ""Jo""], ""B"": [""Po"", ""Jo"", ""J...",GPT,gpt-5.1-2025-11-13,2,988,11,0,none,NaN
25,q001,Arabic,B,C,False,NaN,B,level 6,إذا قررت تهاني شراء السيارة D وإعادة بيعها بع...,"{""A"": ""1575"", ""B"": ""8925"", ""C"": ""9000"", ""D"": ""...",GPT,gpt-5.1-2025-11-13,2,147,11,0,none,NaN
26,q002,Arabic,B,C,False,NaN,B,level 6,في حالة استمرار المبيعات في هذا الاتجاه، في أي...,"{""A"": ""2015"", ""B"": ""2018"", ""C"": ""2020"", ""D"": ""...",GPT,gpt-5.1-2025-11-13,2,129,11,0,none,NaN
34,q010,Arabic,C,A,False,NaN,C,level 6,بالنظر في الفترتين الزمنيتين: 2005 إلى 2010...,"{""A"": [""الهند"", ""كولومبيا ""], ""B"": [""الهند"", ""...",GPT,gpt-5.1-2025-11-13,2,537,11,0,none,NaN
42,q018,Arabic,C,A,False,NaN,C,level 5,هل تمثّل العبارة التالية حقيقة أم رأي؟\nتُعتبر...,"{""A"": [""رأي"", ""حقيقة"", ""حقيقة"", ""رأي""], ""B"": [...",GPT,gpt-5.1-2025-11-13,2,1359,11,0,none,NaN
43,q019,Arabic,C,A,False,NaN,C,level 3,هل يمكن أن تمثّل هذه الجملة الهدف من المقال؟\n...,"{""A"": [""نعم"", ""نعم"", ""لا""], ""B"": [""نعم"", ""لا"",...",GPT,gpt-5.1-2025-11-13,2,797,11,0,none,NaN


In [75]:
df_summary = (
    df_false
    .groupby(['qid'])
    .size()
    .reset_index(name='num_incorrect')
)

print(df_summary)


     qid  num_incorrect
0   q001            172
1   q002            130
2   q003              4
3   q004             79
4   q007              7
5   q008             16
6   q010            162
7   q011              6
8   q012             13
9   q013              4
10  q014              4
11  q015             17
12  q018            140
13  q019            145
14  q023             14
15  q024              4
